In [ ]:
import pandas as pd
import numpy as np
import joblib

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
model_path = "/content/drive/MyDrive/welding_quality_model.pkl"

final_model = joblib.load(model_path)

print("Trained welding quality model loaded successfully.")

Trained welding quality model loaded successfully.


In [ ]:
New_model_off = pd.DataFrame([{
    "Current": 190,
    "Voltage": 24,
    "Travel_Speed": 350,
    "Wire_Feed": 6.5,
    "Torch_Angle": 15,
    "Gap": 4.5,
    "Thickness": 5
}])

display(New_model_off)

,Current,Voltage,Travel_Speed,Wire_Feed,Torch_Angle,Gap,Thickness
0,190,24,350,6.5,15,4.5,5


# New Job Data

In [ ]:
new_job = pd.DataFrame([{
    "Current": 190,
    "Voltage": 24,
    "Travel_Speed": 350,
    "Wire_Feed": 6.5,
    "Torch_Angle": 15,
    "Gap": 4.5,
    "Thickness": 5
}])

print("NEW JOB PARAMETERS")
print("=" * 40)

display(new_job)

NEW JOB PARAMETERS


,Current,Voltage,Travel_Speed,Wire_Feed,Torch_Angle,Gap,Thickness
0,190,24,350,6.5,15,4.5,5


# Get the path followed by the new job through the Decision Tree

In [ ]:
node_indicator = final_model.decision_path(new_job)
leaf_id = final_model.apply(new_job)

tree = final_model.tree_

node_indices = node_indicator.indices[
    node_indicator.indptr[0]:
    node_indicator.indptr[1]
]

print("NEW JOB DECISION PATH")
print("=" * 50)


features = new_job.columns.tolist()

for node_id in node_indices:


    if node_id == leaf_id[0]:
        continue

    feature_index = tree.feature[node_id]

    if feature_index >= 0:

        feature_name = features[feature_index]
        threshold = tree.threshold[node_id]
        value = new_job[feature_name].iloc[0]

        if value <= threshold:
            relation = "<="
        else:
            relation = ">"

        print(
            f"{feature_name}: "
            f"{value} {relation} {threshold:.2f}"
        )

NEW JOB DECISION PATH
Gap: 4.5 > 3.31
Gap: 4.5 > 3.44


In [ ]:
print("\nWHY THE NEW JOB WAS CLASSIFIED AS FAIL")

for node_id in node_indices:

    if node_id == leaf_id[0]:
        continue

    feature_index = tree.feature[node_id]

    if feature_index >= 0:

        feature_name = features[feature_index]
        threshold = tree.threshold[node_id]
        value = new_job[feature_name].iloc[0]

        if value > threshold:

            print(
                f"Risk factor: {feature_name}"
            )

            print(
                f"Observed value: {value}"
            )

            print(
                f"Learned boundary: {threshold:.2f}"
            )

            print(
                f"Reason: {feature_name} "
                f"({value}) is above the learned "
                f"decision boundary ({threshold:.2f})."
            )


WHY THE NEW JOB WAS CLASSIFIED AS FAIL
Risk factor: Gap
Observed value: 4.5
Learned boundary: 3.31
Reason: Gap (4.5) is above the learned decision boundary (3.31).
Risk factor: Gap
Observed value: 4.5
Learned boundary: 3.44
Reason: Gap (4.5) is above the learned decision boundary (3.44).


In [ ]:
prediction = final_model.predict(new_job)[0]

if prediction == 1:
    result = "FAIL"
else:
    result = "PASS"


print("       WELD QUALITY ASSESSMENT")


print(f"Prediction        : {result}")

if result == "FAIL":
    print("Primary risk      : Gap Size")
    print(f"Observed Gap      : {new_job['Gap'].iloc[0]} mm")
    print(f"Learned Boundary  : {threshold:.2f} mm")
    print("Decision Reason   : Gap exceeds learned PASS/FAIL boundary")
else:
    print("Primary risk      : None identified by decision path")



       WELD QUALITY ASSESSMENT
Prediction        : FAIL
Primary risk      : Gap Size
Observed Gap      : 4.5 mm
Learned Boundary  : 3.44 mm
Decision Reason   : Gap exceeds learned PASS/FAIL boundary


System tells from previous learned data that what should be done to **PASS** the job

In [ ]:
pass_jobs = df[df["Result"] == "Pass"].copy()
fail_jobs = df[df["Result"] == "Fail"].copy()

print("PASS jobs:", len(pass_jobs))
print("FAIL jobs:", len(fail_jobs))

PASS jobs: 34
FAIL jobs: 16


In [ ]:
print("HISTORICAL PASS OPERATING RANGE")

display(
    pass_jobs[features]
    .agg(["min", "max", "mean"])
    .round(2)
)

HISTORICAL PASS OPERATING RANGE


,Current,Voltage,Travel_Speed,Wire_Feed,Torch_Angle,Gap,Thickness
min,170.00,22.00,280.00,5.50,10.00,1.00,3.00
max,230.00,27.00,420.00,8.00,20.00,3.38,8.00
mean,200.37,24.44,332.28,6.78,15.77,2.38,5.05


# closest PASS jobs to the new job
this step tells how the new job can be pass after further improvisation
basically when we get to know that the model is failed i.e J021 is fail,we dont directly change parameters instead we take help of old dataset learnings and then estimate the final result

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

scaler = StandardScaler()

pass_scaled = scaler.fit_transform(
    pass_jobs[features]
)

new_job_scaled = scaler.transform(
    new_job[features]
)

distances = pairwise_distances(
    new_job_scaled,
    pass_scaled,
    metric="euclidean"
)[0]

pass_comparison = pass_jobs.copy()
pass_comparison["Distance_to_New_Job"] = distances

nearest_pass = (
    pass_comparison
    .sort_values("Distance_to_New_Job")
    .head(5)
)

display(
    nearest_pass[
        ["Job_ID"] + features + ["Distance_to_New_Job"]
    ]
)

,Job_ID,Current,Voltage,Travel_Speed,Wire_Feed,Torch_Angle,Gap,Thickness,Distance_to_New_Job
32,S013,197.50,24.00,340.0,6.700,16.25,3.375,4.25,1.933717
27,S008,202.50,24.25,335.0,6.850,16.25,3.375,4.25,2.141739
25,S006,197.50,24.00,342.5,6.725,15.00,3.125,4.00,2.261035
33,S014,192.50,24.00,340.0,6.500,18.75,3.125,4.75,2.500380
37,S018,191.25,24.00,337.5,6.450,18.75,2.875,5.00,2.814657


In [ ]:

comparison = nearest_pass[
    ["Job_ID"] + features
].copy()

print("NEW JOB")
display(new_job)

print("\nCLOSEST HISTORICAL PASS JOBS")
display(comparison)

NEW JOB


,Current,Voltage,Travel_Speed,Wire_Feed,Torch_Angle,Gap,Thickness
0,190,24,350,6.5,15,4.5,5



CLOSEST HISTORICAL PASS JOBS


,Job_ID,Current,Voltage,Travel_Speed,Wire_Feed,Torch_Angle,Gap,Thickness
32,S013,197.50,24.00,340.0,6.700,16.25,3.375,4.25
27,S008,202.50,24.25,335.0,6.850,16.25,3.375,4.25
25,S006,197.50,24.00,342.5,6.725,15.00,3.125,4.00
33,S014,192.50,24.00,340.0,6.500,18.75,3.125,4.75
37,S018,191.25,24.00,337.5,6.450,18.75,2.875,5.00


# What is Difference between failed model and New Pass Model

In [ ]:
mean_difference = {}

for feature in features:
    mean_difference[feature] = (
        nearest_pass[feature].mean()
        - new_job[feature].iloc[0]
    )

mean_difference_df = (
    pd.DataFrame.from_dict(
        mean_difference,
        orient="index",
        columns=["Average_Difference"]
    )
    .sort_values("Average_Difference")
)

display(mean_difference_df.round(3))

,Average_Difference
Travel_Speed,-11.000
Gap,-1.325
Thickness,-0.550
Voltage,0.050
Wire_Feed,0.145
Torch_Angle,2.000
Current,6.250


# Search Gap + Travel Speed combinations

In [ ]:
current_speed = new_job["Travel_Speed"].iloc[0]

candidate_gaps = [4.0, 3.8, 3.6, 3.5, 3.4, 3.3, 3.2, 3.0]

candidate_speeds = [
    350, 360, 370, 380, 390, 400, 410, 420, 430, 440, 450
]

results = []

for gap in candidate_gaps:
    for speed in candidate_speeds:

        candidate = new_job.copy()

        candidate["Gap"] = gap
        candidate["Travel_Speed"] = speed

        prediction = final_model.predict(candidate)[0]
        probability = final_model.predict_proba(candidate)[0]

        productivity_improvement = (
            (speed / current_speed) - 1
        ) * 100

        results.append({
            "Gap": gap,
            "Travel_Speed": speed,
            "Prediction": "FAIL" if prediction == 1 else "PASS",
            "PASS_Probability": probability[0],
            "FAIL_Probability": probability[1],
            "Productivity_Improvement_%": productivity_improvement
        })

candidate_results = pd.DataFrame(results)

display(candidate_results.round(3))

,Gap,Travel_Speed,Prediction,PASS_Probability,FAIL_Probability,Productivity_Improvement_%
0,4.0,350,FAIL,0.0,1.0,0.000
1,4.0,360,FAIL,0.0,1.0,2.857
2,4.0,370,FAIL,0.0,1.0,5.714
3,4.0,380,FAIL,0.0,1.0,8.571
4,4.0,390,FAIL,0.0,1.0,11.429
...,...,...,...,...,...,...
83,3.0,410,PASS,1.0,0.0,17.143
84,3.0,420,PASS,1.0,0.0,20.000
85,3.0,430,PASS,1.0,0.0,22.857
86,3.0,440,PASS,1.0,0.0,25.714


# Best Pass Configuaration

In [ ]:
pass_candidates = candidate_results[
    candidate_results["Prediction"] == "PASS"
].copy()

pass_candidates = pass_candidates.sort_values(
    "Travel_Speed",
    ascending=False
)

if len(pass_candidates) > 0:

    best_candidate = pass_candidates.iloc[0]


    print("   Best Pass Configuaration")
    print("")

    print(f"Recommended Gap       : {best_candidate['Gap']} mm")
    print(f"Recommended Speed     : {best_candidate['Travel_Speed']} mm/min")
    print(f"Prediction            : {best_candidate['Prediction']}")
    print(f"PASS Probability      : {best_candidate['PASS_Probability']*100:.2f}%")
    print(f"FAIL Probability      : {best_candidate['FAIL_Probability']*100:.2f}%")
    print(f"Productivity Increase : {best_candidate['Productivity_Improvement_%']:.2f}%")

else:
    print("No PASS configuration found.")

   Best Pass Configuaration

Recommended Gap       : 3.3 mm
Recommended Speed     : 450 mm/min
Prediction            : PASS
PASS Probability      : 100.00%
FAIL Probability      : 0.00%
Productivity Increase : 28.57%


**Final Prediction**

In [ ]:
recommended_job = new_job.copy()

recommended_job["Gap"] = best_candidate["Gap"]
recommended_job["Travel_Speed"] = best_candidate["Travel_Speed"]

final_prediction = final_model.predict(recommended_job)[0]
final_probability = final_model.predict_proba(recommended_job)[0]

final_result = "FAIL" if final_prediction == 1 else "PASS"
final_confidence = max(final_probability)


print(" FINAL COBOT RECOMMENDATION")
print("")


print(f"Original Gap          : {new_job['Gap'].iloc[0]} mm")
print(f"Recommended Gap       : {recommended_job['Gap'].iloc[0]} mm")

print(f"Original Speed        : {new_job['Travel_Speed'].iloc[0]} mm/min")
print(f"Recommended Speed     : {recommended_job['Travel_Speed'].iloc[0]} mm/min")

print(f"\nFinal Prediction      : {final_result}")
print(f"PASS Probability      : {final_probability[0]*100:.2f}%")
print(f"FAIL Probability      : {final_probability[1]*100:.2f}%")
print(f"Confidence            : {final_confidence*100:.2f}%")

print(
    f"Productivity Increase : "
    f"{best_candidate['Productivity_Improvement_%']:.2f}%"
)



 FINAL COBOT RECOMMENDATION

Original Gap          : 4.5 mm
Recommended Gap       : 3.3 mm
Original Speed        : 350 mm/min
Recommended Speed     : 450 mm/min

Final Prediction      : PASS
PASS Probability      : 100.00%
FAIL Probability      : 0.00%
Confidence            : 100.00%
Productivity Increase : 28.57%


# Final J021 assessment

In [ ]:
initial_prediction = final_model.predict(new_job)[0]
initial_probability = final_model.predict_proba(new_job)[0]

initial_result = "FAIL" if initial_prediction == 1 else "PASS"
initial_confidence = max(initial_probability)


print(" INITIAL NEW JOB ASSESSMENT")
print("")


print(f"Prediction       : {initial_result}")
print(f"PASS Probability : {initial_probability[0] * 100:.2f}%")
print(f"FAIL Probability : {initial_probability[1] * 100:.2f}%")
print(f"Confidence       : {initial_confidence * 100:.2f}%")

print("\nPrimary Risk     : Gap Size")
print(f"Observed Gap     : {new_job['Gap'].iloc[0]} mm")
print("Learned Boundary : 3.44 mm")



 INITIAL NEW JOB ASSESSMENT

Prediction       : FAIL
PASS Probability : 0.00%
FAIL Probability : 100.00%
Confidence       : 100.00%

Primary Risk     : Gap Size
Observed Gap     : 4.5 mm
Learned Boundary : 3.44 mm


In [ ]:
baseline_jobs_per_day = 50
target_jobs_per_day = 65

required_improvement = (
    (target_jobs_per_day - baseline_jobs_per_day)
    / baseline_jobs_per_day
) * 100


print(" PRODUCTIVITY TARGET")
print("")


print(f"Current Productivity : {baseline_jobs_per_day} jobs/day")
print(f"Target Productivity  : {target_jobs_per_day} jobs/day")
print(f"Required Improvement : {required_improvement:.1f}%")



 PRODUCTIVITY TARGET

Current Productivity : 50 jobs/day
Target Productivity  : 65 jobs/day
Required Improvement : 30.0%


# best feasible configuration

In [ ]:
if len(pass_candidates) > 0:

    best_candidate = pass_candidates.iloc[0]

    recommended_gap = best_candidate["Gap"]
    recommended_speed = best_candidate["Travel_Speed"]


    print("OPTIMIZED COBOT CONFIGURATION")
    print("")

    print(f"Original Gap          : {new_job['Gap'].iloc[0]} mm")
    print(f"Recommended Gap       : {recommended_gap} mm")

    print(f"\nOriginal Travel Speed : {new_job['Travel_Speed'].iloc[0]} mm/min")
    print(f"Recommended Speed     : {recommended_speed} mm/min")

    print(
        f"\nModel Prediction      : "
        f"{best_candidate['Prediction']}"
    )

    print(
        f"PASS Probability      : "
        f"{best_candidate['PASS_Probability'] * 100:.2f}%"
    )

    print(
        f"FAIL Probability      : "
        f"{best_candidate['FAIL_Probability'] * 100:.2f}%"
    )

    print(
        f"Speed-based Increase : "
        f"{best_candidate['Productivity_Improvement_%']:.2f}%"
    )



else:
    print("No PASS configuration was found.")

OPTIMIZED COBOT CONFIGURATION

Original Gap          : 4.5 mm
Recommended Gap       : 3.3 mm

Original Travel Speed : 350 mm/min
Recommended Speed     : 450 mm/min

Model Prediction      : PASS
PASS Probability      : 100.00%
FAIL Probability      : 0.00%
Speed-based Increase : 28.57%


# Download Model

In [ ]:
import joblib

model_path = "/content/drive/MyDrive/welding_quality_model_final.pkl"

joblib.dump(final_model, model_path)

print("Model exported successfully!")
print(model_path)

Model exported successfully!
/content/drive/MyDrive/welding_quality_model_final.pkl
